# PINO Baseline for PM2.5 Nowcasting (Ronit, Preprocessing-Roadmap Aligned)

This notebook now uses the same preprocessing stack as `exp_04_fno.ipynb` via `src.data`: feature-wise log transforms, grid-wise scaler normalization, cyclic temporal encodings, sparsity masks, derived physical indices, and preprocessing-consistent train/test loading.

In [4]:
# Section 1: Kaggle Runtime Bootstrap (Ronit Paths)
import os, sys

# Kaggle input mounting
KAGGLE_SRC_DATASET = "ronit-pm25-src"
KAGGLE_DATA_ROOT = "/kaggle/input/datasets/khushisingh942004/aisehack"
RONIT_CANDIDATES = [
    f"/kaggle/input/{KAGGLE_SRC_DATASET}",
    f"/kaggle/input/datasets/ronitraj1/{KAGGLE_SRC_DATASET}",
]
DATA_DIR = KAGGLE_DATA_ROOT
CKPT_DIR = "/kaggle/temp"
OUT_DIR = "/kaggle/working"

if os.path.exists('/kaggle'):
    os.environ['AISEHACK_DATA'] = DATA_DIR
    SRC_ROOT = next((p for p in RONIT_CANDIDATES if os.path.exists(os.path.join(p, 'src'))), RONIT_CANDIDATES[0])
else:
    SRC_ROOT = os.path.abspath('../Ronit')
    DATA_DIR = os.path.abspath('../aisehack-theme-2')
    CKPT_DIR = os.path.abspath('./temp')
    OUT_DIR = os.path.abspath('./working')

if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

print(f"SRC_ROOT: {SRC_ROOT}")
print(f"AISEHACK_DATA: {os.environ.get('AISEHACK_DATA', 'not set')}")
print(f"DATA_DIR: {DATA_DIR}")
print(f"CKPT_DIR: {CKPT_DIR}")
print(f"OUT_DIR: {OUT_DIR}")
assert os.path.exists(os.path.join(SRC_ROOT, 'src')), "Could not locate src/ under SRC_ROOT."

SRC_ROOT: /kaggle/input/datasets/ronitraj1/ronit-pm25-src
AISEHACK_DATA: /kaggle/input/datasets/khushisingh942004/aisehack
DATA_DIR: /kaggle/input/datasets/khushisingh942004/aisehack
CKPT_DIR: /kaggle/temp
OUT_DIR: /kaggle/working


In [ ]:
import os

WANDB_API_KEY_DIRECT = "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"

try:
    import wandb
    api_key = WANDB_API_KEY_DIRECT.strip() or os.environ.get('WANDB_API_KEY', '').strip()
    if api_key:
        os.environ['WANDB_API_KEY'] = api_key
        wandb.login(key=api_key, relogin=True)
        print('W&B login complete (using direct API key).')
    else:
        print('No W&B API key found. Proceeding without explicit W&B login.')
except Exception as e:
    print(f'W&B login skipped/failed: {e}')
    print('Training can still continue without W&B syncing.')

In [ ]:
# Section 2: Repository Import, Seed, and Config Load (Roadmap-aligned preprocessing)
import random
import numpy as np
import torch
from src.config import load_config
from src.utils import seed_everything, print_device_info
from src.train import init_wandb, finish_wandb, PERSISTENCE_RMSE_PHYS

cfg = load_config()

# Ensure data path is correct for loader
cfg['paths']['data'] = DATA_DIR

# --- Enforce preprocessing roadmap (from preprocessing.txt + exp_04 pattern) ---
cfg.setdefault('preprocessing', {})
cfg['preprocessing']['normalization'] = 'grid_log_standardize'
cfg['preprocessing']['log_eps'] = 1.0e-12
cfg['preprocessing']['log1p_features'] = ['cpm25', 'rain', 'ventilation_index']
cfg['preprocessing']['log_eps_features'] = ['PM25', 'SO2', 'NOx', 'NH3', 'NMVOC_e', 'NMVOC_finn', 'bio']
cfg['preprocessing']['signed_log_features'] = ['u10', 'v10', 'wind_convergence']
cfg['preprocessing']['sparse_mask_features'] = ['rain', 'NMVOC_finn']
cfg['preprocessing']['sparse_mask_threshold'] = 0.0
cfg['preprocessing']['derived_features'] = ['ventilation_index', 'wind_convergence']
cfg['preprocessing']['feature_time_limits'] = {'cpm25': int(cfg['time']['t_in_cpm'])}

# Enable aux channels so cyclic clocks / masks / derived indices are actually used
cfg['features']['use_aux'] = True
cfg['features']['aux'] = ['lat', 'lon', 'hour_sin', 'hour_cos', 'doy_sin', 'doy_cos']

# Rebuild feature lists exactly like config pipeline does
cfg['features']['all'] = ['cpm25'] + cfg['features']['met'] + cfg['features']['emis']
cfg['features']['base'] = list(cfg['features']['all'])
for name in cfg['preprocessing']['derived_features']:
    if name not in cfg['features']['aux']:
        cfg['features']['aux'].append(name)
for feat in cfg['preprocessing']['sparse_mask_features']:
    mask_name = f"{feat}_mask"
    if mask_name not in cfg['features']['aux']:
        cfg['features']['aux'].append(mask_name)
cfg['features']['input'] = cfg['features']['base'] + cfg['features']['aux']
cfg['features']['n_features'] = len(cfg['features']['base'])
cfg['features']['input_channels'] = len(cfg['features']['input'])

# W&B run identity
cfg.setdefault('wandb', {})
cfg['wandb']['enabled'] = True
cfg['wandb']['run_name'] = 'exp05-pino'

seed_everything(cfg['training']['seed'])
print_device_info(cfg['device'])

print(f"Batch size: {cfg['training']['batch_size_train']}")
print(f"Learning rate: {cfg['training']['lr']}")
print(f"Scheduler: {cfg['training'].get('scheduler', 'coswr')} (T0={cfg['training'].get('t0_epochs', 10)} epochs, T_mult={cfg['training'].get('t_mult', 2)})")
print(f"Epochs: {cfg['training']['epochs']}  |  Patience: {cfg['training']['patience']}")
print(f"Forecast horizon: {cfg['time']['t_out']}")
print(f"lambda_d: {cfg['training']['lambda_d']}")
print(f"lambda_p: {cfg['training']['lambda_p']}")
print(f"Normalization: {cfg['preprocessing']['normalization']}")
print(f"Input channels: {cfg['features']['input_channels']} (base={len(cfg['features']['base'])}, aux={len(cfg['features']['aux'])})")
print(f"W&B: {cfg['wandb']['enabled']} | {cfg['wandb'].get('entity','ronitraj')}/{cfg['wandb'].get('project','aisehack')} | run={cfg['wandb']['run_name']}")


Device: cuda
GPU: Tesla P100-PCIE-16GB
GPU Memory: 17.1 GB
Batch size: 4
Learning rate: 0.0005
Scheduler: coswr (T0=10 epochs, T_mult=2)
Epochs: 50  |  Patience: 15
Forecast horizon: 16
lambda_d: 1.0
lambda_p: 0.02


In [ ]:
# Section 3: Load bounds + grid scaler + months (exp_04-style preprocessing path)
from src.data import (
    load_minmax_bounds,
    build_grid_stats,
    load_grid_stats,
    describe_grid_stats,
    load_all_months,
    get_dataloaders,
    denormalize_cpm25,
    build_test_input,
)
import os
import gc

bounds = load_minmax_bounds(cfg)

# Kaggle safety: scaler path must be writable
if os.path.exists('/kaggle') and str(cfg['paths']['grid_stats']).startswith('/kaggle/input/'):
    cfg['paths']['grid_stats'] = os.path.join('/kaggle/working', os.path.basename(cfg['paths']['grid_stats']))

if not os.path.exists(cfg['paths']['grid_stats']):
    grid_stats = build_grid_stats(cfg, bounds=bounds)
else:
    grid_stats = load_grid_stats(cfg)

cfg.setdefault('_runtime', {})['grid_stats'] = grid_stats
print(f"Loaded grid scaler for {len(grid_stats)} standardized features")
describe_grid_stats(grid_stats, cfg['features']['input'])

print('Using months:', cfg['data']['months'])
print("Loading + normalizing training months ...")
train_data = load_all_months(cfg, cfg['data']['train_months'], bounds)
print("Loading + normalizing validation month ...")
val_data = load_all_months(cfg, [cfg['data']['val_month']], bounds)

train_dl, val_dl = get_dataloaders(cfg, train_data, val_data, bounds)
xb, yb = next(iter(train_dl))
print(f"Train batch x shape: {tuple(xb.shape)}")
print(f"Train batch y shape: {tuple(yb.shape)}")

del train_data, val_data
gc.collect()


Using months: ['APRIL_16', 'JULY_16', 'OCT_16', 'DEC_16']
  Loading APRIL_16 ... OK
  Loading JULY_16 ... OK
  Loading OCT_16 ... OK
  Loading DEC_16 ... OK
Sample month keys: ['cpm25', 'pblh', 't2', 'u10', 'v10', 'q2', 'rain', 'PM25'] ...
Base tensor shape (float32): (4, 10, 715, 140, 124)  (1.99 GB)
Tensor after physical features: (4, 22, 715, 140, 124)  (4.37 GB)
Window-level validation split: 0.10


In [ ]:
# Section 4: Preprocessing checkpoint
print("Roadmap preprocessing is active: log transforms + grid scaler + cyclic aux + sparse masks + derived indices.")
print(f"Normalizer: {cfg['preprocessing']['normalization']}")
print(f"Derived features: {cfg['preprocessing']['derived_features']}")
print(f"Sparse masks: {cfg['preprocessing']['sparse_mask_features']}")
import gc; gc.collect()


Engineered channels already in tensor — skipping duplicate computation.


0

In [ ]:
# Section 6: Build PINO model with preprocessing-aware channel config
from src.model import build_model

print("Batch shape before model build:", tuple(xb.shape))
print(f"Input channels from cfg: {cfg['features']['input_channels']}")
print(f"Temporal window (t_in_met): {cfg['time']['t_in_met']}")

model = build_model(cfg)
print(f"Model type: {cfg['model']['type']}")
print("Model build complete.")

Tensor shape before DataLoader windows: (4, 22, 715, 140, 124)
Configured T_model: 16
Model input channels (C*T_model): 352
Model type: physics_frno


In [9]:
# Section 7: Patch Murtuza/src/model.py: Circular/Reflect Padding Update
# Confirm model uses circular padding for Conv2d and spectral blocks
print('Model padding mode:', cfg.get('model', {}).get('padding_mode', 'circular'))

Model padding mode: circular


In [10]:
# Section 8: Local definition for compute_physics_loss
import torch

def compute_physics_loss(pred, xb, yb, cfg):
    if pred.ndim != 4:
        return torch.zeros((), device=pred.device, dtype=pred.dtype)

    dt = 1.0
    last_c = xb[:, 0, -1, :, :].unsqueeze(-1)
    dC_dt = (pred - last_c) / dt

    grad_h = torch.gradient(pred, dim=1)[0]
    grad_w = torch.gradient(pred, dim=2)[0]

    feature_names = cfg.get('features', {}).get('all', [])
    if not feature_names:
        feature_names = cfg['features']['base'] + ['wind_speed', 'wind_dir', 'RH', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'lat', 'lon', 'lag_1', 'lag_3', 'lag_6']

    def _feat_idx(name: str):
        if name in feature_names:
            idx = feature_names.index(name)
            if idx < xb.shape[1]:
                return idx
        return None

    u_idx = _feat_idx('u10')
    v_idx = _feat_idx('v10')
    if u_idx is None or v_idx is None:
        advection = torch.zeros_like(pred)
    else:
        u = xb[:, u_idx, -1, :, :].unsqueeze(-1)
        v = xb[:, v_idx, -1, :, :].unsqueeze(-1)
        advection = u * grad_h + v * grad_w

    kappa = cfg.get('physics', {}).get('diffusivity', 1.0)
    laplacian = torch.gradient(grad_h, dim=1)[0] + torch.gradient(grad_w, dim=2)[0]
    diffusion = kappa * laplacian

    S = torch.zeros_like(pred)
    for name in ('SO2', 'NOx', 'PM25', 'cpm25'):
        idx = _feat_idx(name)
        if idx is not None:
            S = S + xb[:, idx, -1, :, :].unsqueeze(-1)

    R = dC_dt + advection - diffusion - S
    return torch.mean(R ** 2)

print('compute_physics_loss function ready for PINO training.')


compute_physics_loss function ready for PINO training.


In [11]:
# Section 8: Local definition for compute_physics_loss
import torch

def compute_physics_loss(pred, xb, yb, cfg):
    if pred.ndim != 4:
        return torch.zeros((), device=pred.device, dtype=pred.dtype)

    dt = 1.0
    last_c = xb[:, 0, -1, :, :].unsqueeze(-1)
    dC_dt = (pred - last_c) / dt

    grad_h = torch.gradient(pred, dim=1)[0]
    grad_w = torch.gradient(pred, dim=2)[0]

    feature_names = cfg.get('features', {}).get('all', [])
    if not feature_names:
        feature_names = cfg['features']['base'] + ['wind_speed', 'wind_dir', 'RH', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'lat', 'lon', 'lag_1', 'lag_3', 'lag_6']

    def _feat_idx(name: str):
        if name in feature_names:
            idx = feature_names.index(name)
            if idx < xb.shape[1]:
                return idx
        return None

    u_idx = _feat_idx('u10')
    v_idx = _feat_idx('v10')
    if u_idx is None or v_idx is None:
        advection = torch.zeros_like(pred)
    else:
        u = xb[:, u_idx, -1, :, :].unsqueeze(-1)
        v = xb[:, v_idx, -1, :, :].unsqueeze(-1)
        advection = u * grad_h + v * grad_w

    kappa = cfg.get('physics', {}).get('diffusivity', 1.0)
    laplacian = torch.gradient(grad_h, dim=1)[0] + torch.gradient(grad_w, dim=2)[0]
    diffusion = kappa * laplacian

    S = torch.zeros_like(pred)
    for name in ('SO2', 'NOx', 'PM25', 'cpm25'):
        idx = _feat_idx(name)
        if idx is not None:
            S = S + xb[:, idx, -1, :, :].unsqueeze(-1)

    R = dC_dt + advection - diffusion - S
    return torch.mean(R ** 2)

print('compute_physics_loss function ready for PINO training.')


compute_physics_loss function ready for PINO training.


In [12]:
# Section 10: Patch Murtuza/config.yaml: Add lambda_p, lambda_d, and PINO Hyperparameters
print(f"lambda_d: {cfg['training']['lambda_d']}")
print(f"lambda_p: {cfg['training']['lambda_p']}")
print('PINO hyperparameters loaded from config.yaml.')

lambda_d: 1.0
lambda_p: 0.02
PINO hyperparameters loaded from config.yaml.


In [ ]:
# Section 10b: Dataloader verification (already created in Section 3)
print('DataLoaders are sourced from src.data.get_dataloaders (exp_04 style).')
print(f"Train batches: {len(train_dl)} | Val batches: {len(val_dl)}")
xb_chk, yb_chk = next(iter(train_dl))
print(f"Sample x: {tuple(xb_chk.shape)} | y: {tuple(yb_chk.shape)}")


KeyError: 't_in'

In [ ]:
print("Current train batch shape:", tuple(xb.shape))
print("Current val batch shape:", tuple(next(iter(val_dl))[0].shape))

In [ ]:
# Optional diagnostics (preprocessed dataloader path)
print("Expected met window (t_in_met):", cfg['time'].get('t_in_met', 26))
print("Expected cpm history (t_in_cpm):", cfg['time'].get('t_in_cpm', 10))
print("Expected forecast horizon (t_out):", cfg['time'].get('t_out', 16))
print("Active input feature count:", cfg['features']['input_channels'])

In [ ]:
# W&B — initialise run (same pattern as exp_04)
try:
    import wandb as _wb
    if _wb.run is not None:
        _wb.finish()
except Exception:
    pass

wandb_run = init_wandb(cfg)  # returns None when disabled or login failed
print(f"W&B run: {wandb_run.name if wandb_run else 'disabled/offline'}")

In [ ]:
# Section 11: PINO Train Loop with W&B logging
import numpy as np

PERSISTENCE_RMSE_NORM = 0.0208

def _horizon_weights(cfg, target: torch.Tensor) -> torch.Tensor:
    t_out = target.shape[-1]
    lo = cfg.get('loss', {}).get('horizon_weight_min', 0.8)
    hi = cfg.get('loss', {}).get('horizon_weight_max', 1.4)
    return torch.linspace(lo, hi, t_out, device=target.device, dtype=target.dtype)

def rmse_loss(pred, target, cfg=None):
    spatial_mse = torch.mean((pred - target) ** 2, dim=(1, 2))
    if cfg is not None:
        weights = _horizon_weights(cfg, target)[None]
        spatial_mse = spatial_mse * weights
    return torch.mean(torch.sqrt(spatial_mse + 1e-8))

def mae_loss(pred, target, cfg=None):
    spatial_mae = torch.mean(torch.abs(pred - target), dim=(1, 2))
    if cfg is not None:
        weights = _horizon_weights(cfg, target)[None]
        spatial_mae = spatial_mae * weights
    return torch.mean(spatial_mae)

def objective_loss(pred, target, cfg):
    wrmse = rmse_loss(pred, target, cfg)
    wmae = mae_loss(pred, target, cfg)
    a = cfg.get('loss', {}).get('rmse_weight', 0.8)
    b = cfg.get('loss', {}).get('mae_weight', 0.2)
    return a * wrmse + b * wmae

def rmse_loss_plain(pred, target):
    spatial_mse = torch.mean((pred - target) ** 2, dim=(1, 2))
    return torch.mean(torch.sqrt(spatial_mse + 1e-8))

def get_optimizer(cfg, model, steps_per_epoch):
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg['training']['lr'], weight_decay=cfg['training']['weight_decay'])
    T_0 = max(1, steps_per_epoch * cfg['training'].get('t0_epochs', 10))
    T_mult = cfg['training'].get('t_mult', 2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=T_0, T_mult=T_mult, eta_min=cfg['training']['lr'] * 1e-2)
    return optimizer, scheduler

def train_pino(cfg, model, train_dl, val_dl, bounds=None, wandb_run=None):
    device = cfg['device']
    epochs = cfg['training']['epochs']
    patience = cfg['training'].get('patience', 8)
    grad_clip = cfg['training']['grad_clip']
    save_path = cfg['paths']['model_save']
    lambda_d = cfg['training'].get('lambda_d', 1.0)
    lambda_p = cfg['training'].get('lambda_p', 0.1)

    # Physical-space denorm range for reporting phys RMSE
    cpm_range = None
    if bounds is not None and 'cpm25' in bounds:
        fmin, fmax = bounds['cpm25']
        cpm_range = fmax - fmin

    optimizer, scheduler = get_optimizer(cfg, model, len(train_dl))

    best_val_loss = float('inf')
    patience_count = 0
    history = {
        'train_loss': [], 'val_loss': [],
        'train_objective': [], 'val_phys_rmse': [],
        'selection_metric': [],
    }

    # Log config hyperparams to W&B
    if wandb_run is not None:
        wandb_run.config.update({
            'model_type': cfg['model']['type'],
            'epochs': epochs,
            'lr': cfg['training']['lr'],
            'batch_size_train': cfg['training']['batch_size_train'],
            'lambda_d': lambda_d,
            'lambda_p': lambda_p,
            'input_channels': cfg['features']['input_channels'],
            't_in_met': cfg['time']['t_in_met'],
            't_out': cfg['time']['t_out'],
            'normalization': cfg['preprocessing']['normalization'],
        })

    print(f"\n{'─'*60}")
    print(f"  Persistence gate  (normalized RMSE): {PERSISTENCE_RMSE_NORM:.4f}")
    print(f"  Persistence gate  (physical RMSE):   {PERSISTENCE_RMSE_PHYS:.2f} µg/m³")
    print(f"{'─'*60}\n")

    for epoch in range(epochs):
        model.train()
        epoch_losses = []
        epoch_rmse = []
        for xb, yb in train_dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            data_loss = objective_loss(pred, yb, cfg)
            physics_loss = compute_physics_loss(pred, xb, yb, cfg)
            loss = lambda_d * data_loss + lambda_p * physics_loss
            if not torch.isfinite(loss):
                optimizer.zero_grad()
                continue
            rmse_metric = rmse_loss_plain(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            scheduler.step()
            epoch_losses.append(loss.item())
            epoch_rmse.append(rmse_metric.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_dl:
                xb, yb = xb.to(device), yb.to(device)
                pred = model(xb)
                val_losses.append(rmse_loss_plain(pred, yb).item())

        train_loss = float(np.mean(epoch_rmse))
        val_loss   = float(np.mean(val_losses))
        val_phys   = val_loss * cpm_range if cpm_range else None

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_objective'].append(float(np.mean(epoch_losses)))
        if val_phys is not None:
            history['val_phys_rmse'].append(val_phys)
        history['selection_metric'].append(val_loss)

        # ── W&B per-epoch logging ─────────────────────────────────────────
        if wandb_run is not None:
            log_dict = {
                'epoch': epoch + 1,
                'train/rmse_norm': train_loss,
                'train/objective': float(np.mean(epoch_losses)),
                'val/rmse_norm': val_loss,
                'lr': optimizer.param_groups[0]['lr'],
            }
            if val_phys is not None:
                log_dict['val/rmse_phys'] = val_phys
                log_dict['persistence/ratio'] = val_phys / PERSISTENCE_RMSE_PHYS
            wandb_run.log(log_dict, step=epoch + 1)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_count = 0
            torch.save(model.state_dict(), save_path)
            if wandb_run is not None:
                wandb_run.summary['best_val_rmse_norm'] = best_val_loss
                if val_phys is not None:
                    wandb_run.summary['best_val_rmse_phys'] = val_phys
        else:
            patience_count += 1

        phys_str = f" | Phys: {val_phys:.2f} µg/m³" if val_phys else ""
        print(f"Epoch {epoch+1}/{epochs} | Train: {train_loss:.4f} | Val: {val_loss:.4f}{phys_str} | Best: {best_val_loss:.4f}")
        if patience_count >= patience:
            print(f"Early stopping at epoch {epoch+1}.")
            break

    return history

history = train_pino(cfg, model, train_dl, val_dl, bounds=bounds, wandb_run=wandb_run)

import matplotlib.pyplot as plt
epochs_x = list(range(1, len(history['val_loss']) + 1))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

ax = axes[0]
ax.plot(epochs_x, history['train_loss'], label='Train RMSE (norm)', alpha=0.8)
ax.plot(epochs_x, history['val_loss'],   label='Val RMSE (norm)',   alpha=0.9)
ax.set_xlabel('Epoch'); ax.set_ylabel('Normalized RMSE')
ax.set_title('Training Curves — Normalized Space')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
if history.get('val_phys_rmse'):
    ax.plot(epochs_x, history['val_phys_rmse'], label='Val RMSE (µg/m³)', alpha=0.9)
ax.axhline(PERSISTENCE_RMSE_PHYS, color='red', linestyle='--', label=f'Persistence ({PERSISTENCE_RMSE_PHYS:.2f} µg/m³)')
ax.set_xlabel('Epoch'); ax.set_ylabel('RMSE (µg/m³)')
ax.set_title('Physical-Space Validation RMSE')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[2]
ax.plot(epochs_x, history.get('train_objective', history['train_loss']), label='Train objective', alpha=0.8)
ax.plot(epochs_x, history.get('selection_metric', history['val_loss']), label='Selection metric', alpha=0.9)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Objective / Selection Metric')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(os.path.abspath('../outputs/images'), 'training_curves_pino.png'), dpi=120, bbox_inches='tight')
plt.show()

best_idx = int(np.argmin(history['selection_metric']))
print(f"\n{'─'*55}")
print(f"  Best val RMSE (norm):  {history['val_loss'][best_idx]:.6f} (epoch {best_idx + 1})")
if history.get('val_phys_rmse'):
    best_phys = history['val_phys_rmse'][best_idx]
    print(f"  Best val RMSE (phys):  {best_phys:.3f} µg/m³")
    print(f"  vs Persistence:        {best_phys / PERSISTENCE_RMSE_PHYS:.3f}  {'✓ beats persistence' if best_phys < PERSISTENCE_RMSE_PHYS else '✗ above persistence'}")
print(f"{'─'*55}")


In [ ]:
# Section 12: Notebook Cell: Checkpoint Save/Load Compatible with exp_01_baseline.ipynb
import torch

# Save model checkpoint
torch.save(model.state_dict(), os.path.join(CKPT_DIR, 'best_model.pt'))
print('Model checkpoint saved.')

# Load and sanity check
model.load_state_dict(torch.load(os.path.join(CKPT_DIR, 'best_model.pt'), map_location=cfg['device']))
print('Model checkpoint loaded and verified.')

In [ ]:
# Section 13: Reliable checkpoint load for inference
import os
import shutil
import torch

os.makedirs(OUT_DIR, exist_ok=True)

ckpt_temp = os.path.join(CKPT_DIR, 'best_model.pt')
ckpt_work = os.path.join(OUT_DIR, 'best_model.pt')

if os.path.exists(ckpt_temp):
    shutil.copy2(ckpt_temp, ckpt_work)
    ckpt_to_load = ckpt_work
    print(f"Checkpoint mirrored to working: {ckpt_work}")
elif os.path.exists(ckpt_work):
    ckpt_to_load = ckpt_work
    print(f"Using existing checkpoint in working: {ckpt_work}")
else:
    torch.save(model.state_dict(), ckpt_work)
    ckpt_to_load = ckpt_work
    print(f"No checkpoint file found; saved current model weights to: {ckpt_work}")

state = torch.load(ckpt_to_load, map_location=cfg['device'])
model.load_state_dict(state)
model.eval()
print(f"Checkpoint loaded: {ckpt_to_load}")


In [ ]:
from src.data import build_test_input, denormalize_cpm25

# Notebook-only inference using the same preprocessing as training (src.data pipeline)
device = cfg['device']
n_test = cfg['data']['test_samples']
batch_size = cfg['training']['batch_size_test']
all_preds = []

# Expected channel/time dimensions from model config
expected_c = cfg['features']['input_channels']
expected_t = cfg['time']['t_in_met']

with torch.no_grad():
    for i in range(0, n_test, batch_size):
        j = min(i + batch_size, n_test)
        x_batch = build_test_input(cfg, bounds, start=i, end=j)  # (B, C, T, H, W) in preprocessed space

        if x_batch.shape[1] != expected_c:
            raise RuntimeError(f"Test batch channels {x_batch.shape[1]} != expected {expected_c}")
        if x_batch.shape[2] != expected_t:
            raise RuntimeError(f"Test batch time dim {x_batch.shape[2]} != expected {expected_t}")

        if i == 0:
            print(f"Test batch shape: {x_batch.shape} (chunked mode)")

        batch = torch.from_numpy(x_batch).to(device)
        pred_norm = model(batch)
        pred_phys = denormalize_cpm25(pred_norm.cpu().numpy(), bounds, cfg)
        pred_phys = np.clip(pred_phys, 0.0, None)
        all_preds.append(pred_phys)

preds = np.concatenate(all_preds, axis=0).astype(np.float32)
print(f"Output shape: {preds.shape} | range: [{preds.min():.1f}, {preds.max():.1f}] µg/m³")
assert preds.shape == (n_test, 140, 124, 16), f"Unexpected shape: {preds.shape}"
assert np.isfinite(preds).all(), "Predictions contain NaN/Inf!"

print('Autoregressive 16-hour forecasting complete.')

In [ ]:
# Section 14: Reliable prediction export with verification (atomic save)
import os
import time
import numpy as np

os.makedirs(OUT_DIR, exist_ok=True)

if 'preds' not in globals() or preds is None:
    raise RuntimeError('`preds` is missing. Run Cell 13 first.')

preds = np.asarray(preds, dtype=np.float32)
if not np.isfinite(preds).all():
    raise RuntimeError('`preds` contains NaN/Inf; aborting save.')

final_path = os.path.join(OUT_DIR, 'preds.npy')
tmp_path = final_path + '.tmp'
backup_path = os.path.join(OUT_DIR, f"preds_backup_{int(time.time())}.npy")

# Save temp -> fsync -> atomic replace
with open(tmp_path, 'wb') as f:
    np.save(f, preds)
    f.flush()
    os.fsync(f.fileno())
os.replace(tmp_path, final_path)

# Optional backup copy for extra safety
np.save(backup_path, preds)

# Read-back verification
loaded = np.load(final_path, mmap_mode='r')
assert loaded.shape == preds.shape, f"Saved shape mismatch: {loaded.shape} vs {preds.shape}"

print(f'Predictions saved to {final_path}')
print(f'Backup copy saved to {backup_path}')
print(f'Final file size: {os.path.getsize(final_path) / (1024**2):.2f} MB')

In [ ]:
# Section 14: Proxy Scores + W&B Finish
import numpy as np

# ── Load last-observed cpm25 from test window ────────────────────────────────
_test_cpm25_path = os.path.join(cfg['paths']['data'], 'test_in', 'cpm25.npy')
_test_cpm25_hist = np.load(_test_cpm25_path)          # (N, T_hist, H, W)
last_obs = _test_cpm25_hist[:, -1, :, :]              # (N, H, W)

# ── Align preds shape: expect (N, H, W, T_out) or (N, T_out, H, W) ──────────
if preds.ndim == 4 and preds.shape[1] == cfg['time']['t_out']:
    # (N, T_out, H, W) → (N, H, W, T_out)
    preds_hw_last = preds.transpose(0, 2, 3, 1)
elif preds.ndim == 4 and preds.shape[-1] == cfg['time']['t_out']:
    preds_hw_last = preds                              # already (N, H, W, T_out)
else:
    preds_hw_last = preds

h1_pred  = preds_hw_last[..., 0]                      # (N, H, W)
t_out    = preds_hw_last.shape[-1]
persist  = np.repeat(last_obs[..., None], t_out, axis=-1)  # (N, H, W, T_out)

rmse_h1_vs_last        = float(np.sqrt(np.mean((h1_pred  - last_obs) ** 2)))
mae_h1_vs_last         = float(np.mean(np.abs (h1_pred  - last_obs)))
rmse_all_vs_persistence= float(np.sqrt(np.mean((preds_hw_last - persist) ** 2)))
mae_all_vs_persistence = float(np.mean(np.abs (preds_hw_last - persist)))
mean_abs_step_change   = float(np.mean(np.abs (np.diff(preds_hw_last, axis=-1)))) if t_out > 1 else 0.0

print(f"\n{'─'*60}")
print(f"  Proxy Scores (physical-space, µg/m³)")
print(f"{'─'*60}")
print(f"  RMSE  (H+1  vs last observed):        {rmse_h1_vs_last:.4f}")
print(f"  MAE   (H+1  vs last observed):        {mae_h1_vs_last:.4f}")
print(f"  RMSE  (all horizons vs persistence):  {rmse_all_vs_persistence:.4f}")
print(f"  MAE   (all horizons vs persistence):  {mae_all_vs_persistence:.4f}")
print(f"  Mean abs step change (H→H+1):         {mean_abs_step_change:.4f}")
print(f"{'─'*60}")

# ── Log proxy scores to W&B ──────────────────────────────────────────────────
try:
    if wandb_run is not None:
        wandb_run.log({
            'inference/proxy_rmse_h1_vs_last':      rmse_h1_vs_last,
            'inference/proxy_mae_h1_vs_last':       mae_h1_vs_last,
            'inference/proxy_rmse_all_vs_persist':  rmse_all_vs_persistence,
            'inference/proxy_mae_all_vs_persist':   mae_all_vs_persistence,
            'inference/mean_abs_step_change':       mean_abs_step_change,
        })
        wandb_run.summary['inference/proxy_rmse_h1_vs_last']     = rmse_h1_vs_last
        wandb_run.summary['inference/proxy_rmse_all_vs_persist'] = rmse_all_vs_persistence
        print("Proxy scores logged to W&B.")
except Exception as _e:
    print(f"W&B proxy log skipped: {_e}")

# ── Save training curves artefact to W&B ────────────────────────────────────
try:
    if wandb_run is not None:
        _curves_path = os.path.join(os.path.abspath('../outputs/images'), 'training_curves_pino.png')
        if os.path.exists(_curves_path):
            import wandb as _wb
            wandb_run.log({'training_curves': _wb.Image(_curves_path)})
except Exception as _e:
    print(f"W&B artefact log skipped: {_e}")

# ── Finish W&B run ───────────────────────────────────────────────────────────
finish_wandb(wandb_run, history, bounds, cfg)
print("W&B run finished." if wandb_run is not None else "W&B disabled — run not started.")
